In [ ]:
from pathlib import Path
import importlib

BASE_DIR = Path.cwd().resolve()
MODULE_PATH = BASE_DIR.parent / "online-detection" / "main.py"

online_detection = importlib.machinery.SourceFileLoader(
    "online-detection", str(MODULE_PATH)
).load_module()

In [ ]:
from itertools import islice
from river import datasets, metrics
from river.metrics.report import ClassificationReport

def handle_report(news, **kwargs):
	"""handle_report is called periodically from the pipeline execution, with the most-recent data chunk extractable from the news object."""

	# class_report: ClassificationReport = news.get("class_report")
	precision: metrics.Precision = news.get("precision")
	recall: metrics.Recall = news.get("recall")
	f1: metrics.F1 = news.get("f1")

	print("\n")
	print(precision)
	print(recall)
	print(f1)

def get_data_splits(val_split_index):
	dataset = datasets.CreditCard()

	validation_dataset = islice(dataset, 0, val_split_index)
	inference_dataset = islice(dataset, val_split_index, None)
	return validation_dataset, inference_dataset

In [ ]:
VAL_SPLIT_INDEX = 3963

In [ ]:
validation_dataset, inference_dataset = get_data_splits(val_split_index=VAL_SPLIT_INDEX)

online_detection.pipeline(
    dataset=inference_dataset,
    threshold_tuning_params=None,
    report_every_seconds_elapsed=8*60*60,
    report_callback=handle_report,
    verb=True
)

In [ ]:
validation_dataset, inference_dataset = get_data_splits(val_split_index=VAL_SPLIT_INDEX)
threshold_tuning_params = online_detection.ColdStartThresholdTuningParams(labeled_data=validation_dataset)

online_detection.pipeline(
    dataset=inference_dataset,
    threshold_tuning_params=threshold_tuning_params,
    report_every_seconds_elapsed=8*60*60,
    report_callback=handle_report,
    verb=True
)